# LLM evaluation - Arm B

Author: Mathias Husted

**IMPORTANT:** Check each run with `ollama ps` to make sure Ollama is running on the CPU.

In [1]:
# Imports
import pandas as pd

In [2]:
BENCHMARK_PATH = "../../data/dataset/benchmark.csv"
RUNS_PATH = "../../results/llm"

benchmark = pd.read_csv(BENCHMARK_PATH, dtype=str, keep_default_na=False)


In [3]:
def calc_tps(count_sum, duration_sum):
    return count_sum / (duration_sum / 1e9)

In [4]:
arms = ['A', 'B']
models = ['gemma4_e2b'] # As per file name in /runs
runs_data = [
    {
        'arm': arm,
        'model': model,
        'df': pd.read_csv(f"{RUNS_PATH}/eco-id-results-{arm}-{model}.csv", dtype=str, keep_default_na=False),
        'tps': None,
    }
    for arm in arms
    for model in models
]

# Check file rows match up with benchmark after run
# Then add the ID column to the run data


for run in runs_data:
    df = run["df"]
    assert len(df) == len(benchmark), f"The dataframe {run["arm"]}-{run["model"]} does not have length 319. Check the benchmark matches up"
    # Calculate the average token per seconds rate
    count = df.loc[:, "eval_count"].astype(int)
    duration = df.loc[:, "eval_duration"].astype(int)
    tps = calc_tps(count.sum(), duration.sum())
    run["tps"] = tps
    df.insert(0, "id", None)
    for idx, row in df.iterrows():
        bench_row = benchmark.iloc[idx]
        # Add the ID
        df.loc[idx, 'id'] = bench_row["id"]
        assert bench_row["input"] == row["Input Entity"]



In [8]:
# We create a merged data frame like in the previous evaluation

for run in runs_data:
    required_columns = {"Arm", "Input Entity", "Resolved Name", "Entity Type", "Status"}
    if run["arm"] == "A":
        required_columns.add("Identifier")
    missing = required_columns - set(run['df'].columns)
    assert not missing, f"{run["arm"]}-{run["model"]}: missing columns {missing}"

# Fourth, we join on the IDs and check that no inputs have changed
gold_columns = ["id", "input", "canonical_name", "entity_type", "noise_type", "id_status", "accepted_id"]

for item in runs_data:
    prediction_columns = ["id", "Arm", "Input Entity", "Resolved Name", "Entity Type", "Status"]
    if item['arm'] == "A":
        prediction_columns.append("Identifier")

    item['merged'] = pd.merge(
        left=item['df'][prediction_columns],
        right=benchmark[gold_columns],
        on="id",
        how="left",
        suffixes=("_result", "_benchmark"),
        validate="one_to_one"
    )
    #assert (item['merged']['input_result'] == item['merged']['input_benchmark']).all(), f"b2{item["type"]}: inputs different from benchmark.csv, result file is stale"

runs_data[0]["merged"]

,id,Arm,Input Entity,Resolved Name,Entity Type,Status,Identifier,input,canonical_name,entity_type,noise_type,id_status,accepted_id
0,CHEM-001,A,glucosse,glucose,chemical,resolved,678,glucosse,D-glucose,chemical,typo,ambiguous,5793
1,CHEM-002,A,ureaa,urea,chemical,resolved,300000,ureaa,urea,chemical,typo,single,1176
2,CHEM-003,A,hydroxyureaa,hydroxyurea,chemical,resolved,6111,hydroxyureaa,hydroxyurea,chemical,typo,single,3657
3,CHEM-004,A,cytossine,cytosine,chemical,resolved,6151,cytossine,cytosine,chemical,typo,single,597
4,CHEM-005,A,sperrmine,spermidine,chemical,resolved,5111,sperrmine,spermine,chemical,typo,single,1103
...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,GENE-154,A,MAPK47,MAPK47,gene,resolved,HGNC:11611,MAPK47,,gene,non_existent,none,
315,GENE-155,A,IL73,Interleukin-73,gene,resolved,HGNC:1111,IL73,,gene,non_existent,none,
316,GENE-156,A,SLC6A99,SLC6A9,gene,resolved,HGNC:6111,SLC6A99,,gene,non_existent,none,
317,GENE-157,A,CYP3A88,CYP3A4,gene,resolved,HGNC:10610,CYP3A88,,gene,non_existent,none,
